# Dual WWR CVA Engine

This notebook is the simulation and sensitivity engine of the thesis. It contains only definitions, with no data loading and no results, and it is loaded by `1_main_notebook/dual_wwr_cva_citi_calibration.ipynb` and by `3_grids_9x9/build_citi_joint_grid.py`.

The model combines a Hull-White one-factor short rate $r(t)=x_r(t)+b_r(t)$, a CIR++ diffusive credit intensity $\lambda^{diff}(t)=x_\lambda(t)+b_\lambda(t)$, a constant jump-to-default intensity $\lambda^{jump}$ that enters only through the deterministic margin $u_\tau(t)=Q^{jump}(t)$, and a Gaussian copula that reweights simulated paths by their exposure rank. The evaluation date is 2026-08-11.

| Section | Content |
|---|---|
| 1 | Imports and trade configuration |
| 2 | Market curves and calibration: $b_r(t)$, $b_\lambda(t)$ and the historical estimators |
| 3 | NumPy engine, Modules 1-6: grid, random numbers, bond prices, hazard rates, IRS exposure, $u_\tau(t)$ |
| 4 | PyTorch building blocks for AAD: frozen base curve plus differentiable bump, frozen copula ranks |
| 5 | Conditional AAD: remaining CVA from a realized state $(t_{\tilde k},x_r,x_\lambda)$ |


## 1. Imports and Trade Configuration


In [ ]:
import numpy as np
import QuantLib as ql
import torch
from scipy.interpolate import PchipInterpolator
from scipy.stats import norm
from sklearn.linear_model import LinearRegression

# Baseline trade: 10-year payer IRS (delta = -1), USD 1 million notional, semi-annual payments.
swap_config = {
    'N': 1000000,                      
    'delta': -1,                                     # Payer Swap 
    'payment_dates': [i / 2.0 for i in range(0, 21)] # pay evert half year, 10Y
}

max_years = 10
T_pay = swap_config['payment_dates'] 
notional = swap_config['N']

# Recovery rate and a generic placeholder for the jump-to-default intensity. The
# placeholder is replaced by Citigroup's stress-period calibration by the caller.
recovery_rate = 0.4
S_CDS_ASSUMED = 0.0300
LAMBDA_JUMP_ASSUMED = S_CDS_ASSUMED / (1.0 - recovery_rate)


## 2. Market Curves and Calibration


In [ ]:
# Swaption normal volatilities (bps) used for the Hull-White calibration: (expiry, tenor, vol)
swaption_vols = [
    # Expiry 1.5Y
    (1.5, 1, 111.00), (1.5, 2, 107.40), (1.5, 3, 103.90), (1.5, 4, 100.75),
    (1.5, 5, 98.25), (1.5, 7, 94.05), (1.5, 10, 89.85), (1.5, 30, 83.15),
    
    # Expiry 2Y
    (2.0, 1, 105.95), (2.0, 2, 103.25), (2.0, 3, 100.50), (2.0, 4, 98.25),
    (2.0, 5, 95.90), (2.0, 7, 92.50), (2.0, 10, 88.95), (2.0, 30, 82.80),
    
    # Expiry 3Y
    (3.0, 1, 101.40), (3.0, 2, 98.25), (3.0, 3, 96.30), (3.0, 4, 94.95),
    (3.0, 5, 93.05), (3.0, 7, 90.45), (3.0, 10, 87.95), (3.0, 30, 82.20)
]


In [ ]:
# Calibrate the Hull-White interest model coef. by SOFR historical data. [P-measure: actual probability]
# dr(t) = dx_r(t) + db_r(t). db_r(t) is so small that we can ignore it.
# dr(t) approx= -a r(t) dt + sigma_r*dW_r
def calibrate_hw(sofr_series, dt=1/252):
    dr = sofr_series[1:].values - sofr_series[:-1].values # dr(t) = r(t+1)-r(t)
    r_lag = sofr_series[:-1].values.reshape(-1, 1) # r(t)

    from sklearn.linear_model import LinearRegression
    reg = LinearRegression().fit(r_lag, dr) # dr(t) = intercept + beta*r(t) + e(t)

    a_calibrated = -reg.coef_[0] / dt # beta = -a*dt => a = -beta/dt
    residuals = dr - reg.predict(r_lag) # e(t) = sigma_r*sqrt(dt)*Z_1, where Z_1 ~ N(0,1)
    sigma_r_calibrated = np.std(residuals) / np.sqrt(dt)  # std(e(t)) = sigma_r*sqrt(dt)

    return a_calibrated, sigma_r_calibrated


def get_ql_br_t(market_data, a, sigma_r, max_years=10):
    # 1. Setting
    calculation_date = ql.Date(11, 8, 2026)
    ql.Settings.instance().evaluationDate = calculation_date
    
    # SOFR 
    calendar = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
    day_count = ql.Actual360()
    settlement_days = 2
    
    # 2. SOFR curve (Bootstrapping)
    # market_data = { '1M': xx, '6M': xx, ... }
    helpers = []
    index = ql.Sofr()
    
    for tenor_str, rate in market_data.items():
        tenor = ql.Period(tenor_str)
        helpers.append(ql.OISRateHelper(
            settlement_days,
            tenor,
            ql.QuoteHandle(ql.SimpleQuote(rate)),
            index
        ))
    
    # Log-Cubic curve -> discount curve
    yield_curve = ql.PiecewiseLogCubicDiscount(settlement_days, calendar, helpers, day_count)
    yield_curve.enableExtrapolation()
    
    # 3. b_r(t) 
    # b_r(t) = f_r^M(0,t) + (sigma_r^2 / 2a^2) * (1 - exp(-at))^2
    times = np.linspace(0, max_years, int(max_years * 252))
    br_t_values = []
    
    for t in times:
        # get f_r^M(0, t) 
        f_r_M = yield_curve.forwardRate(t, t, ql.Continuous, ql.NoFrequency).rate()
        
        # Hull-White adjustment
        adjustment_r = (sigma_r**2 / (2 * a**2)) * (1 - np.exp(-a * t))**2
        
        br_t_values.append(f_r_M + adjustment_r)
        
    return times, np.array(br_t_values), yield_curve


def calibrate_cir_regression(l_series, dt=1/252):
    '''Discretized-regression estimator (kept only as an initial guess / cross-check).'''
    l_series = l_series[l_series > 1e-6]
    d_l = l_series[1:].values - l_series[:-1].values
    l_lag = l_series[:-1].values
    y = d_l / np.sqrt(l_lag)
    x1 = 1.0 / np.sqrt(l_lag)
    x2 = np.sqrt(l_lag)
    X = np.column_stack([x1, x2])
    reg = LinearRegression(fit_intercept=False).fit(X, y)
    kappa0 = -reg.coef_[1] / dt
    l_bar0 = (reg.coef_[0] / dt) / kappa0
    res = y - reg.predict(X)
    sigma0 = np.std(res) / np.sqrt(dt)
    return kappa0, l_bar0, sigma0


def get_ql_blambda_t(
    cds_market_data, sofr_yield_curve, kappa, lambda_bar, sigma_lambda, x0, max_years=10, recovery_rate=0.4):
    
    # --- 1. Settings ---
    calc_date = ql.Date(11, 8, 2026)
    ql.Settings.instance().evaluationDate = calc_date
    calendar = ql.UnitedStates(ql.UnitedStates.GovernmentBond)
    day_count = ql.Actual360()
    settlement_days = 2

    # For CDS Bootstrap
    risk_free_ts = ql.RelinkableYieldTermStructureHandle()
    risk_free_ts.linkTo(sofr_yield_curve)

    # --- 2. Market Hazard Curve Bootstrap (QuantLib) ---
    cds_helpers = []
    for tenor_str, spread in cds_market_data.items():
        helper = ql.SpreadCdsHelper(
            ql.QuoteHandle(ql.SimpleQuote(spread)),
            ql.Period(tenor_str),
            settlement_days, calendar, ql.Quarterly, ql.Following,
            ql.DateGeneration.TwentiethIMM, day_count, recovery_rate, risk_free_ts
        )
        cds_helpers.append(helper)

    # Raw Market Data
    hazard_curve = ql.PiecewiseFlatHazardRate(settlement_days, calendar, cds_helpers, day_count)
    hazard_curve.enableExtrapolation()

    # --- 3. PCHIP (Piecewise Cubic Hermite Interpolating Polynomial) ---
    times = np.linspace(0, max_years, int(max_years * 365) + 1)
    
    # Cumulative Hazard Function H(t) = -ln(Q(t))
    raw_H = np.array([-np.log(max(hazard_curve.survivalProbability(float(t)), 1e-16)) for t in times])
    
    # using PCHIP to interpolate the H(t) to obtain the Instantaneous Hazard Rate f_M(0,t)
    H_interp = PchipInterpolator(times, raw_H)
    f_M_smooth = H_interp.derivative()(times)

    # --- 4. f_CIR(0,t) ---
    h = np.sqrt(kappa**2 + 2.0 * sigma_lambda**2)
    f_CIR = []
    
    for t in times:
        if t < 1e-8:
            f_CIR.append(x0)
        else:
            exp_ht = np.exp(h * t)
            denom = 2.0 * h + (kappa + h) * (exp_ht - 1.0)
            term1 = (2.0 * kappa * lambda_bar * (exp_ht - 1.0)) / denom
            term2 = x0 * (4.0 * h * h * exp_ht) / (denom * denom)
            f_CIR.append(term1 + term2)
    
    f_CIR = np.array(f_CIR)

    # --- 5. b_lambda(t) ---
    b_lambda = f_M_smooth - f_CIR

    return {
        "t_axis_l": times,
        "bl_func": PchipInterpolator(times, b_lambda),  
        "fm_curve": PchipInterpolator(times, f_M_smooth),
        "fcir_curve": PchipInterpolator(times, f_CIR),
        "hazard_curve": hazard_curve                 
    }


## 3. NumPy Engine (Modules 1-6)


In [ ]:
def build_time_grid(T, steps_per_year=365):
    """Build the simulation time grid on [0, T] with a fixed number of steps per year."""
    n_steps = int(round(T * steps_per_year))
    t_axis = np.linspace(0.0, T, n_steps + 1)
    dt = np.diff(t_axis)
    return t_axis, dt


def generate_dual_wwr_randoms(t_axis, n_paths, rho_r_lambda, seed=1234, antithetic=True):
    """
    Generate all random inputs shared by the interest-rate and hazard-rate simulators.

    Continuous diffusion WWR (Eq. 5.4): dW_r and dW_lambda are built from two
    independent standard-normal matrices Z1, Z2 via Cholesky-style mixing:
        dW_r(t_k)      = Z1(t_k) * sqrt(dt_k)
        dW_lambda(t_k) = (rho_r_lambda * Z1(t_k) + sqrt(1 - rho_r_lambda^2) * Z2(t_k)) * sqrt(dt_k)

    Jump WWR: u_tau is a single uniform draw per path, constant across time,
    representing that path\'s idiosyncratic tail-event percentile.
    """
    rng = np.random.default_rng(seed)
    dt = np.diff(t_axis)
    n_steps = len(dt)

    if antithetic:
        half = n_paths // 2
        Z1_half = rng.standard_normal((n_steps, half))
        Z2_half = rng.standard_normal((n_steps, half))
        Z1 = np.concatenate([Z1_half, -Z1_half], axis=1)
        Z2 = np.concatenate([Z2_half, -Z2_half], axis=1)
        u_half = rng.uniform(0.0, 1.0, size=half)
        u_tau = np.concatenate([u_half, 1.0 - u_half])
    else:
        Z1 = rng.standard_normal((n_steps, n_paths))
        Z2 = rng.standard_normal((n_steps, n_paths))
        u_tau = rng.uniform(0.0, 1.0, size=n_paths)

    sqrt_dt = np.sqrt(dt)[:, np.newaxis]
    dW_r = Z1 * sqrt_dt
    dW_lambda = (rho_r_lambda * Z1 + np.sqrt(1.0 - rho_r_lambda ** 2) * Z2) * sqrt_dt

    return {"Z1": Z1, "Z2": Z2, "dW_r": dW_r, "dW_lambda": dW_lambda, "u_tau": u_tau}


In [ ]:
def simulate_forward_zcb_paths(yield_curve, a, sigma_r, dW_r, t_axis, payment_dates):
    """
    Simulate zero-coupon bond prices P(t_k, T_h) under the T_m-forward measure Q_{T_m},
    where T_m = payment_dates[-1] is the numeraire maturity.

    Rather than simulating the short rate r(t) directly, this simulates the forward
    bond price FwdP(t_k; T_m, T_h) = P(t_k, T_h) / P(t_k, T_m) as a log-normal
    martingale under Q_{T_m} (Eq. 5.1):

        sigma_P(t, T)   = (sigma_r / a) * (1 - exp(-a * (T - t)))
        Sigma_F(t, T_h) = sigma_P(t, T_h) - sigma_P(t, T_m)

        FwdP(t_k) = FwdP(t_{k-1}) * exp(
            -0.5 * Sigma_F(t_{k-1})^2 * dt - Sigma_F(t_{k-1}) * dW_r(t_k)
        )

    P(t_k, T_h) is then recovered as FwdP(t_k; T_m, T_h) / FwdP(t_k; T_m, t_k).
    """
    payment_dates = np.asarray(payment_dates, dtype=float)
    n_steps = len(t_axis)
    n_paths = dW_r.shape[1]
    n_targets = len(payment_dates)
    T_m = payment_dates[-1]

    P_paths = np.zeros((n_steps, n_paths, n_targets))

    P0_Tm = yield_curve.discount(float(T_m))
    P0_targets = np.array([yield_curve.discount(float(T)) for T in payment_dates])
    FwdP = np.tile(P0_targets / P0_Tm, (n_paths, 1))
    P_paths[0, :, :] = P0_targets

    for k in range(n_steps - 1):
        t_prev, t_curr = t_axis[k], t_axis[k + 1]
        dt = t_curr - t_prev

        active_idx = np.where(payment_dates >= t_curr)[0]
        active_T = payment_dates[active_idx]

        sigma_P_Tm = (sigma_r / a) * (1.0 - np.exp(-a * (T_m - t_prev)))
        sigma_P_active = (sigma_r / a) * (1.0 - np.exp(-a * (active_T - t_prev)))
        Sigma_F = sigma_P_active - sigma_P_Tm

        FwdP[:, active_idx] *= np.exp(
            -0.5 * (Sigma_F ** 2) * dt - Sigma_F * dW_r[k, :, np.newaxis]
        )

        exact_idx = np.where(np.isclose(payment_dates, t_curr))[0]
        if len(exact_idx) > 0:
            FwdP_benchmark = FwdP[:, exact_idx].squeeze(axis=1)
            P_tcurr_Tm = 1.0 / FwdP_benchmark
        else:
            next_idx = np.where(payment_dates > t_curr)[0][0]
            T_R = payment_dates[next_idx]
            FwdP_TR = FwdP[:, next_idx]
            tau = T_R - t_curr
            P_tcurr_TR = np.exp(-yield_curve.forwardRate(t_curr, T_R, ql.Continuous).rate() * tau)
            P_tcurr_Tm = P_tcurr_TR / FwdP_TR

        P_paths[k + 1, :, :] = FwdP * P_tcurr_Tm[:, np.newaxis]

    return P_paths


def path_survival_probabilities(lambda_diff, t_axis):
    """
    Compute the path-wise survival probability Q_j^diff(t_k) and its increment
    Delta Q_j^diff(t_k), using a left-Riemann sum for the integrated hazard:

        Q_j^diff(t_k)       = exp(- sum_{i=1}^{k} lambda_diff_j(t_{i-1}) * dt_i)
        Delta Q_j^diff(t_k) = Q_j^diff(t_k) - Q_j^diff(t_{k-1})   (negative by construction)
    """
    n_steps, n_paths = lambda_diff.shape
    dt = np.diff(t_axis)[:, np.newaxis]

    integrated_hazard = lambda_diff[:-1, :] * dt
    Q = np.ones((n_steps, n_paths))
    Q[1:, :] = np.exp(-np.cumsum(integrated_hazard, axis=0))

    delta_Q = np.diff(Q, axis=0)  # aligned to t_1 .. t_{n_steps-1}
    return Q, delta_Q


# Module 4 - CIR++ hazard-rate simulation (Milstein) with continuous WWR drift
def simulate_hazard_paths(
    kappa, lambda_bar, sigma_lambda, x0,
    a, sigma_r, rho_r_lambda,
    bl_func, dW_lambda, t_axis, T_m,
):
    """
    Simulate the CIR++ shifted square-root process x_lambda(t) via the Milstein
    scheme (Eq. 5.2), including the continuous-WWR drift adjustment that arises
    from the correlation between the rate and hazard Brownian motions:

        mu(x_lambda, t) = kappa * (lambda_bar - x_lambda)
            - (rho_r_lambda * sigma_lambda * sigma_r / a) * (1 - exp(-a*(T_m - t))) * sqrt(x_lambda)

        x_lambda(t_k) = x_lambda(t_{k-1}) + mu(x_lambda, t_{k-1}) * dt
            + sigma_lambda * sqrt(x_lambda(t_{k-1})) * dW_lambda(t_k)
            + 0.25 * sigma_lambda^2 * (dW_lambda(t_k)^2 - dt)

    Values are floored at zero (full truncation) whenever the square root would
    otherwise be undefined. The deterministic CIR++ shift b_lambda(t) is added
    afterwards (Eq. 5.3):

        lambda_diff(t_k) = x_lambda(t_k) + b_lambda(t_k)
    """
    n_steps = len(t_axis)
    n_paths = dW_lambda.shape[1]
    dt = np.diff(t_axis)

    x_lambda = np.zeros((n_steps, n_paths))
    x_lambda[0, :] = x0

    for k in range(n_steps - 1):
        t = t_axis[k]
        x_prev = np.maximum(x_lambda[k], 0.0)
        sqrt_x = np.sqrt(x_prev)

        wwr_term = (rho_r_lambda * sigma_lambda * sigma_r / a) * (1.0 - np.exp(-a * (T_m - t))) * sqrt_x
        drift = (kappa * (lambda_bar - x_prev) - wwr_term) * dt[k]

        diffusion = sigma_lambda * sqrt_x * dW_lambda[k]
        milstein_corr = 0.25 * (sigma_lambda ** 2) * (dW_lambda[k] ** 2 - dt[k])

        x_lambda[k + 1] = np.maximum(x_prev + drift + diffusion + milstein_corr, 0.0)

    lambda_diff = x_lambda + bl_func(t_axis)[:, np.newaxis]
    return lambda_diff


In [ ]:
# Module 5 - IRS exposure valuation
def fair_swap_rate(yield_curve, payment_dates):
    """Par swap rate K such that V(0) = 0: K = (P(0,T0) - P(0,Tm)) / annuity."""
    Tk = np.asarray(payment_dates, dtype=float)
    P0_T0 = yield_curve.discount(float(Tk[0]))
    P0_Tm = yield_curve.discount(float(Tk[-1]))
    alphas = np.diff(Tk)
    P0_Tk = np.array([yield_curve.discount(float(t)) for t in Tk[1:]])
    annuity = np.sum(alphas * P0_Tk)
    return (P0_T0 - P0_Tm) / annuity


def swap_cashflow_weights(payment_dates, K):
    """Fixed cash-flow weight vector: w_0 = -1, w_h = K*alpha_h, w_m = 1 + K*alpha_m."""
    Tk = np.asarray(payment_dates, dtype=float)
    alphas = np.diff(Tk)
    m = len(Tk) - 1
    w = np.zeros(m + 1)
    w[0] = -1.0
    w[1:m] = K * alphas[:-1]
    w[m] = 1.0 + K * alphas[-1]
    return w


def simulate_irs_value(P_paths, t_axis, payment_dates, notional, position_sign, yield_curve):
    """
    Reconstruct the undiscounted path-wise IRS value:

        V_j(t_k) = position_sign * N * ( -1{t_k > T_0} + sum_{T_h >= t_k} w_h * P_j(t_k, T_h) )

    position_sign = +1 for a receiver swap, -1 for a payer swap.
    Returns V_j(t_k) and the positive exposure E_j(t_k) = max(V_j(t_k), 0).

    The active weight vector w_h(t_k) = w_h * 1{T_h >= t_k} is piecewise constant in
    t_k: it only changes at the m+1 payment dates. Rather than looping over every
    simulation step and re-selecting the active columns (an expensive 3D fancy-index
    per step), the time grid is grouped into the runs where the active mask is
    constant and each run is priced with a single batched matrix-vector product.
    """
    n_steps, n_paths, n_targets = P_paths.shape
    Tk = np.asarray(payment_dates, dtype=float)
    t_axis = np.asarray(t_axis)
    K = fair_swap_rate(yield_curve, Tk)
    w = swap_cashflow_weights(Tk, K)

    active_mask = Tk[np.newaxis, :] >= np.round(t_axis, 4)[:, np.newaxis]  # (n_steps, n_targets)
    mask_key = active_mask @ (1 << np.arange(n_targets))
    change_points = np.where(np.diff(mask_key) != 0)[0] + 1
    segment_starts = np.concatenate([[0], change_points])
    segment_ends = np.concatenate([change_points, [n_steps]])

    v_all = np.zeros((n_steps, n_paths))
    for s, e in zip(segment_starts, segment_ends):
        w_active = w * active_mask[s]
        block = P_paths[s:e].reshape(-1, n_targets)
        v_all[s:e] = (block @ w_active).reshape(e - s, n_paths)

    correction = np.where(t_axis > Tk[0], w[0], 0.0)
    v_all += correction[:, np.newaxis]

    V = position_sign * notional * v_all
    E = np.maximum(V, 0.0)
    return V, E, K


def compute_market_implied_u_tau(t_axis, lambda_jump):
    """
    Deterministic, time-varying u_tau(t_k) = Q^jump(t_k) = exp(-lambda_jump * t_k),
    the Credit-Triangle survival probability. Same value for every path at a
    given t_k; varies across t_k, not across paths -- the opposite broadcast
    axis from the original per-path random u_tau.
    """
    return np.exp(-lambda_jump * np.asarray(t_axis))


## 4. PyTorch Building Blocks for AAD


In [ ]:
_TENOR_AAD = {
    "1W": 1/52, "2W": 2/52,
    "1M": 1/12, "2M": 2/12, "3M": 3/12, "6M": 6/12, "9M": 9/12,
    "1Y": 1., "2Y": 2., "3Y": 3., "4Y": 4., "5Y": 5.,
    "7Y": 7., "10Y": 10., "15Y": 15., "20Y": 20., "30Y": 30.,
}


def bump_interp(t_query, bump_pillars, T_pillars):
    """Piecewise-linear interpolation of a small per-pillar bump, anchored at
    0 at t=0. The bump is exactly 0 everywhere when bump_pillars is 0."""
    dtype = bump_pillars.dtype
    T_ext = torch.cat([torch.zeros(1, dtype=dtype), T_pillars])
    b_ext = torch.cat([torch.zeros(1, dtype=dtype), bump_pillars])
    t_c = torch.clamp(t_query, T_ext[0], T_ext[-1])
    idx = torch.clamp(torch.searchsorted(T_ext.contiguous(), t_c.contiguous()) - 1, 0, len(T_ext) - 2)
    alpha = (t_c - T_ext[idx]) / (T_ext[idx + 1] - T_ext[idx] + 1e-14)
    return b_ext[idx] + alpha * (b_ext[idx + 1] - b_ext[idx])


def cir_forward_intensity(t, kappa, lambda_bar, sigma_lambda, x0):
    """Closed-form CIR forward intensity f_lambda^CIR(0,t) (same formula used in
    calibration), differentiable w.r.t. kappa, lambda_bar, sigma_lambda, x0."""
    h = torch.sqrt(kappa ** 2 + 2.0 * sigma_lambda ** 2)
    exp_ht = torch.exp(h * t)
    denom = 2.0 * h + (kappa + h) * (exp_ht - 1.0)
    term1 = (2.0 * kappa * lambda_bar * (exp_ht - 1.0)) / denom
    term2 = x0 * (4.0 * h ** 2 * exp_ht) / (denom ** 2)
    return term1 + term2


In [ ]:
def path_survival_probabilities_torch(lambda_diff, t_axis):
    """Torch-differentiable version of path_survival_probabilities."""
    n_steps, n_paths = lambda_diff.shape
    dtype = lambda_diff.dtype
    dt = torch.as_tensor(np.diff(t_axis), dtype=dtype).unsqueeze(1)

    integrated_hazard = lambda_diff[:-1, :] * dt
    logQ = -torch.cumsum(integrated_hazard, dim=0)
    Q = torch.cat([torch.ones((1, n_paths), dtype=dtype), torch.exp(logQ)], dim=0)
    delta_Q = Q[1:] - Q[:-1]
    return Q, delta_Q


# Module 5 (torch) - IRS exposure valuation on frozen-exact + bump curve
def compute_fixed_K(yield_curve, payment_dates):
    """The swap's fixed (contractual) rate: breakeven under the ORIGINAL,
    unbumped curve. Computed once and held fixed for every subsequent bump --
    see the markdown note above on why Market Delta must be measured against
    an existing contract's fixed rate, not one re-struck under each bump."""
    Tk_np = np.asarray(payment_dates, dtype=float)
    P0_pay = np.array([yield_curve.discount(float(t)) for t in Tk_np])
    alphas = Tk_np[1:] - Tk_np[:-1]
    annuity = np.sum(alphas * P0_pay[1:])
    return float((P0_pay[0] - P0_pay[-1]) / annuity)


def simulate_irs_value_torch(P_paths, t_axis, payment_dates, notional, position_sign, dtype, K_fixed):
    """Torch-differentiable version of Module 5. K_fixed is a plain constant
    (see compute_fixed_K) -- it does NOT depend on bump_R, so Market Delta
    (d CVA/d bump_R) flows entirely through the simulated discount paths
    P_paths (Module 3), matching the standard convention of holding an
    EXISTING trade's fixed rate constant when computing its rate sensitivity."""
    n_steps, n_paths, n_targets = P_paths.shape
    Tk_np = np.asarray(payment_dates, dtype=float)
    Tk = torch.as_tensor(Tk_np, dtype=dtype)
    K = torch.as_tensor(K_fixed, dtype=dtype)

    alphas = Tk[1:] - Tk[:-1]
    m = len(Tk_np) - 1
    w_list = [torch.tensor(-1.0, dtype=dtype)]
    w_list += [K * alphas[i] for i in range(m - 1)]
    w_list += [1.0 + K * alphas[-1]]
    w = torch.stack(w_list)

    active_mask = Tk_np[np.newaxis, :] >= np.round(np.asarray(t_axis), 4)[:, np.newaxis]

    V_list = []
    for k in range(n_steps):
        active_idx = np.where(active_mask[k])[0]
        if len(active_idx) == 0:
            V_list.append(torch.zeros(n_paths, dtype=dtype))
            continue
        idx_t = torch.as_tensor(active_idx, dtype=torch.long)
        v_t = P_paths[k][:, idx_t] @ w[idx_t]
        if t_axis[k] > Tk_np[0]:
            v_t = v_t + w[0]
        V_list.append(v_t)

    V = torch.stack(V_list, dim=0) * (position_sign * notional)
    E = torch.clamp(V, min=0.0)
    return V, E, K


# Module 6 (torch) - Gaussian Copula weights, frozen ranks (Market Delta decoupling)
def frozen_copula_ranks(E_frozen_np, u_tau_t_np, eps=1e-6):
    """
    Precompute the frozen (non-differentiable) Copula rank transform x_E, x_tau
    from a reference realization of the exposure and the market-implied,
    deterministic, time-varying jump-WWR latent variable u_tau_t (Module 6,
    market-implied). Per the Market Delta decoupling convention, these ranks
    are treated as CONSTANTS for every theta except rho_copula.
    """
    n_steps, n_paths = E_frozen_np.shape
    ranks = np.argsort(np.argsort(E_frozen_np, axis=1), axis=1) + 1
    u_E = np.clip(ranks / (n_paths + 1), eps, 1.0 - eps)
    x_E = norm.ppf(u_E)
    x_tau = norm.ppf(np.clip(u_tau_t_np, eps, 1.0 - eps))
    return (
        torch.as_tensor(x_E, dtype=torch.float64),
        torch.as_tensor(x_tau, dtype=torch.float64),
    )


def copula_path_weights_torch(x_E, x_tau, rho_copula, eps=1e-12):
    """Gaussian Copula weights (Module 6, market-implied) as a differentiable
    function of rho_copula ONLY -- x_E, x_tau are frozen constants (see
    frozen_copula_ranks). x_tau is broadcast over the PATHS axis (n_steps, 1),
    the opposite convention from the original per-path-random x_tau (1, n_paths),
    since u_tau_t now varies by time step, not by path. This is what makes
    Term 3 (Copula Channel) vanish for every theta except rho_copula."""
    rho2 = rho_copula ** 2
    x_tau_col = x_tau.unsqueeze(1)
    log_density = -(rho2 * (x_E ** 2 + x_tau_col ** 2) - 2.0 * rho_copula * x_E * x_tau_col) / (2.0 * (1.0 - rho2 + eps))
    density = torch.exp(log_density) / torch.sqrt(1.0 - rho2 + eps)
    return density / torch.sum(density, dim=1, keepdim=True)


# Module 7 (torch) - Final CVA aggregation
def compute_dual_wwr_cva_torch(E, P_paths_Tm, delta_Q_diff, w_copula, P0_Tm, recovery_rate):
    """Torch-differentiable version of Module 7."""
    rebased_exposure = E[1:, :] / P_paths_Tm[1:, :]
    weighted_sum = rebased_exposure * w_copula[1:, :] * delta_Q_diff
    loss_given_default = 1.0 - recovery_rate
    return -loss_given_default * P0_Tm * torch.sum(weighted_sum)


# End-to-end AAD driver
def build_frozen_noise(t_axis, n_paths, seed=1234):
    """Module 2, frozen: draws Z1, Z2 (antithetic) once with NumPy. u_tau_t is no
    longer drawn here -- it is the deterministic market-implied curve from
    compute_market_implied_u_tau, independent of the Monte Carlo seed."""
    n_steps = len(t_axis)
    rng = np.random.default_rng(seed)
    half = n_paths // 2
    Z1_half = rng.standard_normal((n_steps - 1, half))
    Z2_half = rng.standard_normal((n_steps - 1, half))
    Z1 = np.concatenate([Z1_half, -Z1_half], axis=1)
    Z2 = np.concatenate([Z2_half, -Z2_half], axis=1)
    return Z1, Z2


## 5. Conditional AAD


In [ ]:
# Conditional AAD driver: restarts the HW1F/CIR++ simulation at an arbitrary
# (t_start, x_r_start, x_lambda_start) instead of (0, 0, x_lambda_0). Validated
# to reproduce run_aad_sensitivities EXACTLY at t_start=0 (see accompanying
# scratch verification: CVA rel diff -0.000000%, all 8 bucketed deltas match).
def simulate_forward_zcb_paths_torch_conditional(
    yield_curve, a, sigma_r, bump_R, T_pillars_r, dW_r, t_axis, payment_dates, dtype, x_r_start=0.0,
):
    payment_dates_np = np.asarray(payment_dates, dtype=float)
    payment_dates_t = torch.as_tensor(payment_dates_np, dtype=dtype)
    n_steps = len(t_axis)
    n_paths = dW_r.shape[1]
    T_m_local = payment_dates_np[-1]
    t_start = float(t_axis[0])

    def B_hw(t, T):
        return (1.0 - torch.exp(-a * (T - t))) / a

    def phi_hw(t):
        f0t = yield_curve.forwardRate(float(t), float(t), ql.Continuous, ql.Annual).rate()
        return f0t + (sigma_r ** 2 / (2 * a ** 2)) * (1.0 - torch.exp(-a * torch.tensor(t, dtype=dtype))) ** 2

    def A_hw(t, T):
        P0T = yield_curve.discount(float(T))
        P0t = yield_curve.discount(float(t)) if t > 1e-12 else 1.0
        f0t = yield_curve.forwardRate(float(t), float(t), ql.Continuous, ql.Annual).rate()
        Bt = B_hw(t, T)
        a_val = float(a.item())
        return (P0T / P0t) * torch.exp(Bt * f0t - (sigma_r ** 2 / (4 * a)) * (1.0 - np.exp(-2 * a_val * t)) * Bt ** 2)

    x_r_start_t = torch.as_tensor(x_r_start, dtype=dtype)

    def P_t_T_closed(T):
        Bt = B_hw(t_start, T)
        bump_T = bump_interp(torch.tensor([T], dtype=dtype), bump_R, T_pillars_r)[0]
        At = A_hw(t_start, T) * torch.exp(-bump_T * T)
        r_t = x_r_start_t + phi_hw(t_start)
        return At * torch.exp(-Bt * r_t)

    P0_targets = torch.stack([P_t_T_closed(float(T)) for T in payment_dates_np])
    P0_Tm = P_t_T_closed(T_m_local)
    P0_grid = torch.stack([P_t_T_closed(float(t)) for t in t_axis])

    FwdP = (P0_targets / P0_Tm).unsqueeze(0).repeat(n_paths, 1)
    P_paths = [P0_targets.unsqueeze(0).repeat(n_paths, 1)]

    for k in range(n_steps - 1):
        t_prev, t_curr = float(t_axis[k]), float(t_axis[k + 1])
        dt_k = t_curr - t_prev

        active_idx = np.where(payment_dates_np >= t_curr - 1e-9)[0]
        active_idx_t = torch.as_tensor(active_idx, dtype=torch.long)
        active_T = payment_dates_t[active_idx_t]

        sigma_P_Tm = (sigma_r / a) * (1.0 - torch.exp(-a * (T_m_local - t_prev)))
        sigma_P_active = (sigma_r / a) * (1.0 - torch.exp(-a * (active_T - t_prev)))
        Sigma_F = sigma_P_active - sigma_P_Tm

        FwdP_active = FwdP[:, active_idx_t] * torch.exp(
            -0.5 * (Sigma_F ** 2) * dt_k - Sigma_F.unsqueeze(0) * dW_r[k].unsqueeze(1)
        )
        FwdP_new = FwdP.clone()
        FwdP_new[:, active_idx_t] = FwdP_active
        FwdP = FwdP_new

        exact_idx = np.where(np.isclose(payment_dates_np, t_curr))[0]
        if len(exact_idx) > 0:
            FwdP_bench = FwdP[:, int(exact_idx[0])]
            P_tcurr_Tm = 1.0 / FwdP_bench
        else:
            next_idx = int(np.where(payment_dates_np > t_curr)[0][0])
            T_R = payment_dates_t[next_idx]
            FwdP_TR = FwdP[:, next_idx]
            tau = T_R - t_curr
            f_t_TR = (torch.log(P0_grid[k + 1]) - torch.log(P0_targets[next_idx])) / tau
            P_tcurr_TR = torch.exp(-f_t_TR * tau)
            P_tcurr_Tm = P_tcurr_TR / FwdP_TR

        P_paths.append(FwdP * P_tcurr_Tm.unsqueeze(1))

    return torch.stack(P_paths, dim=0), P0_Tm


def simulate_hazard_paths_torch_conditional(
    fm_curve_calibration, kappa, lambda_bar, sigma_lambda, x0,
    a, sigma_r, rho_r_lambda,
    bump_lambda, T_pillars_l, dW_lambda, t_axis, T_m, dtype,
):
    """Same recursion as simulate_hazard_paths_torch (Part 4); uses ELAPSED time
    (t_axis - t_axis[0]) in the CIR closed-form forward intensity so x0 is
    correctly the state AT t_axis[0] (may be > 0). Identical to the original
    when t_axis[0] == 0."""
    n_steps = len(t_axis)
    n_paths = dW_lambda.shape[1]
    dt = np.diff(t_axis)
    t0_offset = float(t_axis[0])

    x_lambda = [x0.expand(n_paths).clone()]
    for k in range(n_steps - 1):
        t = float(t_axis[k])
        x_prev = torch.clamp(x_lambda[-1], min=0.0)
        sqrt_x = torch.sqrt(x_prev)
        wwr_term = (rho_r_lambda * sigma_lambda * sigma_r / a) * (1.0 - torch.exp(-a * (T_m - t))) * sqrt_x
        drift = (kappa * (lambda_bar - x_prev) - wwr_term) * dt[k]
        diffusion = sigma_lambda * sqrt_x * dW_lambda[k]
        milstein_corr = 0.25 * (sigma_lambda ** 2) * (dW_lambda[k] ** 2 - dt[k])
        x_next = torch.clamp(x_prev + drift + diffusion + milstein_corr, min=0.0)
        x_lambda.append(x_next)
    x_lambda = torch.stack(x_lambda, dim=0)

    t_axis_t = torch.as_tensor(np.asarray(t_axis, dtype=float), dtype=dtype)
    fM_frozen = torch.tensor(fm_curve_calibration(t_axis), dtype=dtype)
    fM = fM_frozen + bump_interp(t_axis_t, bump_lambda, T_pillars_l)
    fCIR = cir_forward_intensity(t_axis_t - t0_offset, kappa, lambda_bar, sigma_lambda, x0)
    b_lambda = fM - fCIR
    return x_lambda + b_lambda.unsqueeze(1)


def run_aad_sensitivities_conditional(
    t_start, x_r_start, x_lambda_start,
    yield_curve, fm_curve_calibration, sofr_market_quotes, cds_market_data,
    a0, sigma_r0, kappa0, lambda_bar0, sigma_lambda0,
    rho_r_lambda0, rho_copula0,
    payment_dates, notional, position_sign, recovery_rate,
    T_max, n_paths, seed=1234, steps_per_year=26, dtype=torch.float64,
    lambda_jump_assumed=None,
):
    """Remaining CVA from t_start to T_max, given a REALIZED state at t_start.
    Reduces exactly to run_aad_sensitivities at t_start=0, x_r_start=0,
    x_lambda_start=x_lambda_0."""
    sofr_tenors = list(sofr_market_quotes.keys())
    T_pillars_r = torch.tensor([_TENOR_AAD[t] for t in sofr_tenors], dtype=dtype)
    bump_R = torch.zeros(len(sofr_tenors), dtype=dtype, requires_grad=True)

    cds_tenors = list(cds_market_data.keys())
    T_pillars_l = torch.tensor([_TENOR_AAD[t] for t in cds_tenors], dtype=dtype)
    bump_lambda = torch.zeros(len(cds_tenors), dtype=dtype, requires_grad=True)

    a = torch.tensor(a0, dtype=dtype, requires_grad=True)
    sigma_r = torch.tensor(sigma_r0, dtype=dtype, requires_grad=True)
    kappa = torch.tensor(kappa0, dtype=dtype, requires_grad=True)
    lambda_bar = torch.tensor(lambda_bar0, dtype=dtype, requires_grad=True)
    sigma_lambda = torch.tensor(sigma_lambda0, dtype=dtype, requires_grad=True)
    rho_r_lambda = torch.tensor(rho_r_lambda0, dtype=dtype, requires_grad=True)
    rho_copula = torch.tensor(rho_copula0, dtype=dtype, requires_grad=True)

    if lambda_jump_assumed is None:
        lambda_jump_assumed = LAMBDA_JUMP_ASSUMED

    n_steps = int(round((T_max - t_start) * steps_per_year)) + 1
    t_axis_local = np.linspace(t_start, T_max, n_steps)
    Z1, Z2 = build_frozen_noise(t_axis_local, n_paths, seed=seed)
    u_tau_t_np = compute_market_implied_u_tau(t_axis_local, lambda_jump_assumed)

    dt_np = np.diff(t_axis_local)
    sqrt_dt_t = torch.as_tensor(dt_np ** 0.5, dtype=dtype).unsqueeze(1)
    Z1_t = torch.as_tensor(Z1, dtype=dtype)
    Z2_t = torch.as_tensor(Z2, dtype=dtype)
    dW_r = Z1_t * sqrt_dt_t
    dW_lambda = (rho_r_lambda * Z1_t + torch.sqrt(1.0 - rho_r_lambda ** 2) * Z2_t) * sqrt_dt_t

    T_m_local = payment_dates[-1]
    K_fixed_local = compute_fixed_K(yield_curve, payment_dates)

    P_paths, P0_Tm = simulate_forward_zcb_paths_torch_conditional(
        yield_curve, a, sigma_r, bump_R, T_pillars_r, dW_r, t_axis_local, payment_dates, dtype, x_r_start=x_r_start,
    )
    x_lambda_start_t = torch.tensor(x_lambda_start, dtype=dtype)
    lambda_diff = simulate_hazard_paths_torch_conditional(
        fm_curve_calibration, kappa, lambda_bar, sigma_lambda, x_lambda_start_t, a, sigma_r, rho_r_lambda,
        bump_lambda, T_pillars_l, dW_lambda, t_axis_local, T_m_local, dtype,
    )
    Q_diff, delta_Q_diff = path_survival_probabilities_torch(lambda_diff, t_axis_local)
    V, E, K = simulate_irs_value_torch(P_paths, t_axis_local, payment_dates, notional, position_sign, dtype, K_fixed_local)

    E_frozen_np = E.detach().numpy()
    x_E, x_tau = frozen_copula_ranks(E_frozen_np, u_tau_t_np)
    w_copula = copula_path_weights_torch(x_E, x_tau, rho_copula)

    P_paths_Tm = P_paths[:, :, -1]
    cva = compute_dual_wwr_cva_torch(E, P_paths_Tm, delta_Q_diff, w_copula, P0_Tm, recovery_rate)
    cva.backward()

    return {
        "cva": cva.item(), "bump_R": bump_R, "sofr_tenors": sofr_tenors,
        "bump_lambda": bump_lambda, "cds_tenors": cds_tenors, "sigma_r": sigma_r,
    }
